In [11]:
import redis
r =redis.Redis(
    host="localhost",
    port=6379,
    decode_responses=True,
)
article_id = 1002

In [13]:
views=r.incr(
    f"article:{article_id}:views"
)
print(views)
print(article_id)

2
1002


In [16]:
import redis
import json
import time
def get_user_from_db(user_id):
    print("正在查询数据库...")

    time.sleep(2)  # 模拟数据库耗时

    return {
        "id": user_id,
        "name": "Leon",
        "age": 25,
    }


In [21]:
# 查询用户
def get_user(user_id):

    key = f"user:{user_id}"

    # 1. 查询Redis缓存
    cached = r.get(key)

    if cached:
        print("Redis缓存命中")
        return json.loads(cached)


    # 2. Redis没有，查数据库
    print("Redis没有缓存")

    user = get_user_from_db(user_id)


    # 3. 写入Redis
    r.set(
        key,
        json.dumps(user),
        ex=60,
    )

    return user

user_id = 1001


result = get_user(user_id)

print(result['name'])


Redis没有缓存
正在查询数据库...
Leon


In [22]:
def query_database(user_id):
    print("查询数据库")

    return {
        "id": user_id,
        "name": "Leon",
    }


In [ ]:
def get_user(user_id):

    key = f"user:{user_id}"

    cached = r.get(key)

    if cached:
        print("Redis命中")
        return json.loads(cached)
    print("Redis没有")

    user = query_database(user_id)

    if user:
        r.set(
            key,
            json.dumps(user),
            ex=300,
        )

    return user


In [30]:
def get_user(user_id):

    key = f"user:{user_id}"

    cached = r.get(key)

    if cached == "__NULL__":
        return None

    if cached:
        print("Redis命中")
        return json.loads(cached)
    print("Redis没有")

    user = query_database(user_id)

    if user is None:

        r.set(
            key,
            "__NULL__",
            ex=60,
        )

        return None

    r.set(
        key,
        json.dumps(user),
        ex=300,
    )

    return user

In [32]:
# 调用函数
result = get_user(1002)

print(result)

Redis命中
{'id': 1002, 'name': 'Leon'}
